# Attention Attribution

Attribute model behavior to specific attention patterns: knockout effects,
value decomposition, position-specific attribution, and logit contributions.

References:
- Voita et al. (2019) "Analyzing Multi-Head Self-Attention"
- Clark et al. (2019) "What Does BERT Look At?"

## Why This Matters

Attention attribution reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_attribution import (
    attention_knockout_attribution,
    attention_value_decomposition,
    attention_logit_contribution,
    attention_pattern_metric_sensitivity,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 5, 10, 15, 20, 25, 30])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Attention Knockout Attribution

Zero out each head and measure how much the metric changes.
Positive effect means the head was helping the metric.

In [ ]:
result = attention_knockout_attribution(model, tokens, metric_fn)
print(f"Total attribution: {result['total_attribution']:.4f}")
print(f"\nTop heads (most important):")
for l, h, e in result['top_heads'][:5]:
    print(f"  L{l}H{h}: {e:+.4f}")
print(f"\nBottom heads (inhibitory):")
for l, h, e in result['bottom_heads'][:5]:
    print(f"  L{l}H{h}: {e:+.4f}")

## Attention Value Decomposition

For each head at a given layer, see which source positions contribute
most to the output through the attention-weighted value mechanism.

In [ ]:
for layer in range(2):
    result = attention_value_decomposition(model, tokens, layer=layer)
    print(f"\nLayer {layer}:")
    for head, sources in result['top_sources_per_head'].items():
        top3 = [(pos, f'{val:.3f}') for pos, val in sources[:3]]
        print(f"  Head {head}: {top3}")

## Attention Logit Contribution

Project each head's output through the unembedding to see which tokens it promotes.

In [ ]:
result = attention_logit_contribution(model, tokens)
print("Top tokens promoted by each head:")
for (l, h), tok_list in sorted(result['head_top_tokens'].items()):
    top3 = [(t, f'{v:.3f}') for t, v in tok_list[:3]]
    print(f"  L{l}H{h}: {top3}")

## Attention Pattern Sensitivity

How sensitive is the metric to noise in a specific head's attention pattern?

In [ ]:
for l in range(2):
    for h in range(2):
        result = attention_pattern_metric_sensitivity(model, tokens, metric_fn, l, h)
        print(f"L{l}H{h}: sensitivity={result['sensitivity']:.4f}, max_dev={result['max_deviation']:.4f}")